# WIMP-LEGEND — Poster physics workbench

**Working notebook, 26 October 2026 — preliminary / not independently certified for poster publication.**

This notebook is intended for hands-on analysis, not to replace the modular `wimp_legend` package. It supports four activities:

1. Inspect and vary the **source-resolved background benchmark**.
2. Compare the historical and revised backgrounds **at matched assumptions**.
3. Explore the existing **pilot SI and \(\mathcal O_8/\mathcal O_{11}\)** sensitivity results, retaining their limitations.
4. Run a small **physical recoil-rate smoke test** using actual `wimp_legend` APIs, then develop the final sensitivity driver separately.

**Do not interpret the pilot input as measured LEGEND efficiency, activity, detector geometry or an accepted enriched-Ge isotope mixture.** The optional pilot calculations use a **100% Ge-76 surrogate**, median 90% Poisson-CLs, a 95%-detected-signal ROI and fixed-background assumptions. The newer `legend_poster` background is a *different scenario* from the matched `enriched_UAr` benchmark because cooldown/veto assumptions change.

See `docs/BG_BENCHMARK.md`, `docs/GE_NUCLEAR_DATA.md`, `docs/SI_NORMALIZATION.md`, `docs/SENSITIVITY.md` and the corresponding independent audits for authoritative API/provenance details. The current poster branch is the source of truth.

## 0 · Setup

Extract the provided starter ZIP **at the repository root**, preserving `notebooks/poster_workbench.ipynb` and `notebooks/poster_workbench_assets/`.

Install the repository and notebook dependencies into the same environment as Jupyter:

```bash
python -m pip install -e '.[test,poster]'
python -m pip install jupyter pandas matplotlib
```

Open Jupyter from the repository root (`jupyter lab`) and open `notebooks/poster_workbench.ipynb`. The bundled pilot CSVs can be explored immediately; later replace them with your own generated CSVs. The notebook never overwrites the bundled files.

In [ ]:
from pathlib import Path
import json, subprocess, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Jupyter's working directory is normally the folder where it was launched.
def find_repo_root(start):
    for p in (start, *start.parents):
        if (p / 'pyproject.toml').exists() and (p / 'src' / 'wimp_legend').exists():
            return p
    raise FileNotFoundError('Run Jupyter inside your WIMP-LEGEND checkout, or set ROOT manually.')

ROOT = find_repo_root(Path.cwd().resolve())
ASSETS = ROOT / 'notebooks' / 'poster_workbench_assets'
print('Repository:', ROOT)
print('Git revision:', subprocess.run(['git','rev-parse','--short','HEAD'], cwd=ROOT, capture_output=True, text=True).stdout.strip())
print('Bundled pilot data:', ASSETS.exists())

## 1 · Read benchmark assumptions, *before* comparing curves

The benchmark's activity normalizations, LAr transfer/veto probabilities and response settings are **assumptions**, not measured LEGEND subsystem inputs. Change the versioned configuration for new hypotheses, and keep its provenance with the results.

In [ ]:
CONFIG = ROOT / 'examples' / 'bg_benchmark_config.json'
config = json.loads(CONFIG.read_text(encoding='utf-8'))
print('Status:', config['status'])
print('Analysis threshold [keV observed]:', config['plotting']['threshold_keV'])
print('Energy range [keV observed]:', config['plotting']['min_energy_keV'], 'to', config['plotting']['max_energy_keV'])
print('Baseline efficiency:', config['plotting']['constant_detection_efficiency'])
rows = []
for label, sc in config['scenarios'].items():
    rows.append({'scenario':label, 'cooldown [days]':sc['cooldown_day'],
                 'run [days]':sc['run_duration_day'], 'UAr/AAr activity [Bq/kg Ar]':sc['ar39_activity_Bq_per_kg_Ar'],
                 'LAr veto survival':sc['lar_veto_survival_probability'],
                 'H3 initial [decays/kg/day]':dict(sc['activities_per_kg_day'])['H3']})
display(pd.DataFrame(rows))

## 2 · Generate the background benchmark (optional)

First run the standard package tests after fixing any local line-ending/checksum issues. The plotting command is separate from the regular default-policy public-count integration tests; both must be checked before claiming numerical acceptance.

This cell is **disabled by default** because the full background generation is computationally nontrivial. Set `RUN_BACKGROUND=True` to generate fresh `benchmark-output/component_rates.csv` and figures.

In [ ]:
RUN_BACKGROUND = False
OUTPUT_DIR = ROOT / 'benchmark-output'
if RUN_BACKGROUND:
    subprocess.run([sys.executable, str(ROOT/'examples'/'background_benchmark.py'),
                    '--config', str(CONFIG), '--output', str(OUTPUT_DIR)],
                   cwd=ROOT, check=True)
else:
    print('Using existing background CSV if available; otherwise bundled illustrative pilot data.')

CURRENT_CSV = OUTPUT_DIR / 'component_rates.csv'
BACKGROUND_CSV = CURRENT_CSV if CURRENT_CSV.exists() else ASSETS / 'pilot_component_rates.csv'
bg = pd.read_csv(BACKGROUND_CSV)
print('Loaded:', BACKGROUND_CSV)
print('Scenarios:', sorted(bg['scenario'].unique()))
print('Components:', sorted(bg['component'].unique()))
display(bg.head(4))

## 3 · Plot source-separated energy spectra

Rate CSV units are **events/(kg Ge · day · keV observed)**. For plotting, multiply by 365.25 to obtain events/(kg Ge · Julian year · keV observed). `Total` is already the sum of components; do not add it again.

In [ ]:
BG_SCENARIO = 'legend_poster'             # or enriched_UAr / natural_AAr
ENERGY_VIEW = (0.08, 2.0)               # zoom around low-energy features
component_order = ['Total','H3','Ge68','Ga68','Zn65','Fe55','Ar39','solar_CEvNS','solar_ES','flat_observed']
fig, ax = plt.subplots(figsize=(9,5))
for comp in component_order:
    t = bg[(bg.scenario == BG_SCENARIO) & (bg.component == comp)].sort_values('bin_lower_keV_obs')
    if t.empty: continue
    edges = np.r_[t.bin_lower_keV_obs.to_numpy(), t.bin_upper_keV_obs.to_numpy()[-1]]
    vals = t.events_per_kg_day_keV_obs.to_numpy() * 365.25
    ax.stairs(vals, edges, label=comp, linewidth=2 if comp=='Total' else 1)
ax.set(xlim=ENERGY_VIEW, yscale='log', xlabel='Observed energy [keV]',
       ylabel='Events / (kg Ge year keV observed)', title=f'{BG_SCENARIO} — preliminary source breakdown')
ax.axvline(config['plotting']['threshold_keV'], color='gray', linestyle=':')
ax.legend(ncol=2, fontsize=8); ax.grid(alpha=.25)
plt.show()

## 4 · Compare background models in signal-relevant windows

**Do not compare different detector configurations as if only the background physics changed.** `legacy_enriched_UAr` vs `enriched_UAr` is the matched illustrative comparison. `legend_poster` additionally changes cooldown and veto assumptions.

In [ ]:
def window_counts(frame, scenario, lower, upper, exposure_kg_day=365.25, component='Total'):
    r = frame[(frame.scenario==scenario)&(frame.component==component)].copy()
    # Exact for these analysis boundaries because the supplied grid contains them.
    overlap = np.maximum(0.0, np.minimum(r.bin_upper_keV_obs.to_numpy(), upper) -
                                np.maximum(r.bin_lower_keV_obs.to_numpy(), lower))
    return float(np.dot(r.events_per_kg_day_keV_obs.to_numpy(), overlap) * exposure_kg_day)

rows=[]
for upper in (0.3, 0.5, 1.0, 2.0, 15.0):
    a=window_counts(bg,'legacy_enriched_UAr',0.1,upper)
    b=window_counts(bg,'enriched_UAr',0.1,upper)
    rows.append({'Energy [keV observed]':f'0.1–{upper:g}', 'Legacy events':a,
                 'New events':b, 'New/legacy':b/a if a else np.nan})
display(pd.DataFrame(rows).round(4))

## 5 · Preliminary SI limits from the separate pilot calculation

The bundled results were computed using a **pure 76Ge surrogate**, not the actual enriched-Ge mixture; the response is uncalibrated, and no external data overlay or independent sensitivity validation was performed. Use this to explore the presentation and decide what the next physics calculation must improve.

To generate fresh results, see `LEGEND_WIMP_poster_preliminary_scan_2026-10-26.zip` and its `legend_pilot_sensitivity.py`. Its CSV must be updated if your final background/config differs.

In [ ]:
PILOT_CSV = ASSETS / 'pilot_sensitivity_results.csv'
sens = pd.read_csv(PILOT_CSV)
print('Pilot results:', PILOT_CSV)
print('Available masses:', sorted(sens.mass_GeV.unique()))
print('Operators:', sorted(sens.op.unique()))
display(sens.head(5))

In [ ]:
EXPOSURE_TONNE_YEAR = 1.0
fig, ax = plt.subplots(figsize=(8,5))
for scenario, label in [('legacy_enriched_UAr','Historical B'), ('enriched_UAr','Revised B'),
                        ('legend_poster','Cooldown/veto alternative')]:
    rows=sens[(sens.op==1)&(sens.scenario==scenario)&(sens.exposure_ton_year==EXPOSURE_TONNE_YEAR)].sort_values('mass_GeV')
    ax.plot(rows.mass_GeV, rows.sigma_p_90_cm2, marker='o', label=label)
ax.set(xscale='log',yscale='log',xlabel=r'WIMP mass $m_\chi$ [GeV]',
       ylabel=r'Median 90% CL $\sigma_p$ [cm$^2$]',
       title='Preliminary SI — do not publish as a validated LEGEND projection')
ax.legend(); ax.grid(alpha=.25); plt.show()

## 6 · Preliminary NREFT operator sensitivities

For these single-operator scans, limits are on \(|c_i^p|\) in GeV\(^{-2}\), with equal proton and neutron coefficients and the package's frozen half-isospin convention. They are **not SI cross sections**. The pilot uses \(\mathcal O_8\) and \(\mathcal O_{11}\); later test additional operators/interference and actual isotope abundances.

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
for op in (8, 11):
    for scenario, style in (('enriched_UAr','-'),('legacy_enriched_UAr','--')):
        r=sens[(sens.op==op)&(sens.scenario==scenario)&(sens.exposure_ton_year==EXPOSURE_TONNE_YEAR)].sort_values('mass_GeV')
        ax.plot(r.mass_GeV,r.coefficient_90_GeV_minus2,linestyle=style,label=f'O{op}, {scenario}')
ax.set(xscale='log',yscale='log',xlabel=r'WIMP mass $m_\chi$ [GeV]',
       ylabel=r'Median 90% CL $|c_i^p|$ [GeV$^{-2}$]',
       title='Exploratory single-operator Wilson-coefficient sensitivities')
ax.legend(fontsize=8);ax.grid(alpha=.25);plt.show()

## 7 · Optional: direct physical recoil-spectrum evaluation through `wimp_legend`

This small calculation bypasses the pilot CSV and uses the **actual** Ge response and NREFT rate APIs. It is deliberately off by default because it performs several numerical halo integrations. The isotope mass and composition remain **explicit surrogate values**; their replacement with sourced bare nuclear masses and real enrichment fractions is a separate analysis task. Use this as an educational starting point for the final driver, not as a poster result.

In [ ]:
RUN_SIGNAL_DEMO = False
if RUN_SIGNAL_DEMO:
    from wimp_legend.targets import (Isotope, SourceReference,
        germanium_afh2014_momentum, load_germanium_afh2014)
    from wimp_legend.halo import StandardHaloModel
    from wimp_legend.rates import NREFTRateConfig, differential_rate_per_kg_day_keV
    from wimp_legend.interactions import WilsonCoefficients
    from wimp_legend.si import si_wilson_coefficients

    A = 76  # synthetic single-isotope approximation, NOT an actual enriched composition
    source = SourceReference('Approximate A*u mass for demonstration', 'poster workbench', 'synthetic')
    isotope = Isotope('Ge', A, 32, A*0.93149410242, 0.0,
                      {key:source for key in ('identity','mass_GeV','spin')})
    responses = load_germanium_afh2014(isotope, momentum=germanium_afh2014_momentum(A))
    halo = StandardHaloModel.from_km_s(v0_km_s=220, vesc_km_s=544, v_lab_km_s=266)
    recoil_keV = np.geomspace(0.01, 2.0, 40)
    mchi = 5.0
    inputs = dict(m_chi_GeV=mchi, rho_chi_GeV_cm3=0.3,
        targets_per_kg=1000*6.02214076e23/A, dataset=responses,
        j_chi=0.5, config=NREFTRateConfig(0.938272),
        speed_pdf=halo.lab_speed_pdf, beta_max=halo.max_lab_speed,
        breakpoints=halo.integration_breakpoints, epsrel=1e-7)
    wilson = {
        'O1 (sigma_p=1e-45 cm^2)': si_wilson_coefficients(1e-45,m_chi_GeV=mchi,
                   m_proton_GeV=0.938272,fn_over_fp=1.,proton_sign=1),
        'O8 (c_p=c_n=1e-7 GeV^-2)': WilsonCoefficients.from_proton_neutron({8:(1e-7,1e-7)}),
        'O11 (c_p=c_n=1e-7 GeV^-2)': WilsonCoefficients.from_proton_neutron({11:(1e-7,1e-7)}),
    }
    fig, ax = plt.subplots(figsize=(8,5))
    for label, coefficients in wilson.items():
        rate = differential_rate_per_kg_day_keV(recoil_keV,coefficients=coefficients,**inputs)
        ax.plot(recoil_keV,rate,label=label)
    ax.set(yscale='log',xlabel='True nuclear recoil energy [keVnr]',
           ylabel='Events / (kg day keVnr) at the stated reference coupling')
    ax.grid(alpha=.25);ax.legend(fontsize=8);plt.show()
else:
    print('Set RUN_SIGNAL_DEMO=True to evaluate physical Ge recoil spectra using the package.')

## 8 · Analysis checklist before poster figures are frozen

- [ ] Background corrected source commit and exact source hashes checked on the local `poster` branch; the appropriate independent audit applies.
- [ ] Use **sourced** natural/enriched Ge isotope fractions and bare nuclear masses, not the pilot pure-76Ge surrogate.
- [ ] Agree with supervisor on threshold, analysis acceptance, quenching, resolution, exposure, activity/cooling history and LAr-veto scenario.
- [ ] Reproduce the legacy SI curve using **identical signal and response assumptions** before attributing deviations to the new background.
- [ ] Explicitly select statistical method (exact known-background Poisson CLs counting vs. binned profile), ROI policy, confidence level and nuisance assumptions.
- [ ] Independently validate absolute SI normalization, detector-folded spectra, low-mass kinematic support, binning/ROI stability, and finite-momentum Ge responses (particularly near diffraction minima).
- [ ] Compare individual **sourced** external observed limits and projections with recorded confidence-level and halo assumptions. Avoid importing an anonymous envelope as scientific evidence.
- [ ] Keep at most one or two scientifically interesting NREFT examples; do not relabel Wilson coefficients as conventional SI cross sections.
- [ ] Record `git rev-parse HEAD`, input hashes and analysis parameters for each poster figure.

**Suggested next notebook extension:** wire the physical `SIO1Signal` and selected single-operator truth spectra into the accepted NR response and benchmark-background count model; validate a single 5-GeV point against a separate calculation before producing a mass/exposure grid.